# 09 - E5 Baseline multiclase agrupado sagital

**Objetivo:** construir un primer baseline multiclase 2D sagital sobre SPIDER, partiendo del pipeline binario ya validado.

Este notebook evalúa si una U-Net 2D simple puede diferenciar grupos técnicos de etiquetas de la máscara, en lugar de segmentar solo foreground/background.

**Alcance:** modalidad T1, eje sagital 2, target `256x256`, mapping agrupado preliminar de 4 clases, selección de slice sin máscara con `center_window_best_prediction`.

**Fuera de alcance:** nnU-Net, 3D, axial, backend y nombres anatómicos definitivos sin documentación formal del dataset.


## 1. Instalación e importación de dependencias

In [ ]:
!pip -q install SimpleITK scikit-image tqdm

In [ ]:
from pathlib import Path
import json
import random
import warnings
from typing import Dict, List, Tuple, Any

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.transform import resize
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("SimpleITK:", sitk.Version())


## 2. Verificación de entorno y rutas

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
if DEVICE.type == "cpu":
    print("ADVERTENCIA: no se detectó GPU. El entrenamiento multiclase puede ser lento en CPU.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
BINARY_IMPROVED_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_baseline_mejorado_binario")
HOLDOUT_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_holdout_validacion")
SLICE_EVAL_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_slice_selection_eval")
MULTICLASS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [MULTICLASS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

CANDIDATES_CSV = PREPROCESS_ROOT / "E4_baseline_candidates_no_space.csv"
HOLDOUT_SELECTED_CSV = HOLDOUT_ROOT / "E5_holdout_selected_cases.csv"
BINARY_MODEL_PATH = BINARY_IMPROVED_ROOT / "E5_improved_unet2d_binary_best.pt"
SLICE_SELECTION_REPORT_JSON = SLICE_EVAL_ROOT / "E5_slice_selection_report.json"

print("CANDIDATES_CSV:", CANDIDATES_CSV)
print("BINARY_MODEL_PATH:", BINARY_MODEL_PATH)
print("SLICE_SELECTION_REPORT_JSON:", SLICE_SELECTION_REPORT_JSON)
print("MULTICLASS_ROOT:", MULTICLASS_ROOT)


## 3. Carga de candidatos T1

In [ ]:
MODALITY_FILTER = "t1"
N_CASES = 100
SAGITTAL_AXIS = 2
TARGET_SIZE = (256, 256)
NUM_CLASSES = 4
BASE_CHANNELS = 16
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-3
CENTER_WINDOW_RADIUS = 3
INFERENCE_BATCH_SIZE = 16

candidates_df = pd.read_csv(CANDIDATES_CSV)

def infer_case_modality(row):
    text = " ".join(str(v).lower() for v in row.values)
    if "t2" in text:
        return "t2"
    if "t1" in text:
        return "t1"
    return "unknown"


def ensure_case_id(df):
    out = df.copy()
    if "case_id" not in out.columns:
        for candidate in ["file_name", "image_path", "source_image_path"]:
            if candidate in out.columns:
                out["case_id"] = out[candidate].apply(lambda x: Path(str(x)).stem)
                break
    if "case_id" not in out.columns:
        raise ValueError("No se pudo construir case_id.")
    out["case_id"] = out["case_id"].astype(str)
    return out


candidates_df = ensure_case_id(candidates_df)
if "modality" not in candidates_df.columns:
    candidates_df["modality"] = candidates_df.apply(infer_case_modality, axis=1)

t1_df = candidates_df[candidates_df["modality"].str.lower().eq(MODALITY_FILTER)].copy()
if len(t1_df) == 0:
    raise RuntimeError("No hay candidatos T1 disponibles.")

selected_df = t1_df.sample(n=min(N_CASES, len(t1_df)), random_state=SEED).reset_index(drop=True)
selected_cases_csv_path = MULTICLASS_ROOT / "E5_multiclass_selected_cases.csv"
selected_df.to_csv(selected_cases_csv_path, index=False)

print("Candidatos totales:", len(candidates_df))
print("Candidatos T1:", len(t1_df))
print("Casos seleccionados:", len(selected_df))
print("CSV selección:", selected_cases_csv_path)
display(selected_df.head())


## 4. Análisis de labels originales

In [ ]:
def resolve_path(value):
    return Path(str(value))


def get_case_paths(row):
    image_candidates = ["image_path", "source_image_path", "image", "img_path"]
    mask_candidates = ["mask_path", "source_mask_path", "mask", "seg_path"]
    image_path = None
    mask_path = None

    for column in image_candidates:
        if column in row and pd.notna(row[column]):
            image_path = resolve_path(row[column])
            break

    for column in mask_candidates:
        if column in row and pd.notna(row[column]):
            mask_path = resolve_path(row[column])
            break

    if image_path is None or mask_path is None:
        raise ValueError("No se encontraron columnas de path para imagen/máscara.")

    return image_path, mask_path


def read_mha(path: Path):
    itk_image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(itk_image)
    return itk_image, array


label_counts = {}
label_case_frequency = {}
total_voxels = 0

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Analizando labels"):
    _, mask_path = get_case_paths(row)
    _, mask = read_mha(mask_path)

    labels, counts = np.unique(mask, return_counts=True)
    total_voxels += int(mask.size)
    present_labels = set()

    for label, count in zip(labels, counts):
        label_int = int(label)
        label_counts[label_int] = label_counts.get(label_int, 0) + int(count)
        present_labels.add(label_int)

    for label_int in present_labels:
        label_case_frequency[label_int] = label_case_frequency.get(label_int, 0) + 1

foreground_voxels = sum(count for label, count in label_counts.items() if label != 0)
label_rows = []

for label in sorted(label_counts):
    label_rows.append({
        "original_label": int(label),
        "voxel_count": int(label_counts[label]),
        "case_frequency": int(label_case_frequency.get(label, 0)),
        "voxel_ratio_total": float(label_counts[label] / max(total_voxels, 1)),
        "foreground_ratio_within_foreground": float(label_counts[label] / max(foreground_voxels, 1)) if label != 0 else 0.0,
    })

original_label_distribution_df = pd.DataFrame(label_rows)
original_label_distribution_csv_path = MULTICLASS_ROOT / "E5_multiclass_original_label_distribution.csv"
original_label_distribution_df.to_csv(original_label_distribution_csv_path, index=False)

observed_original_labels = sorted(int(x) for x in original_label_distribution_df["original_label"].tolist())

print("Labels originales globales:", observed_original_labels)
print("CSV distribución labels:", original_label_distribution_csv_path)
display(original_label_distribution_df)


## 5. Mapping multiclase agrupado

El mapping siguiente es técnico y preliminar. No asigna nombres anatómicos definitivos porque este notebook no incorpora documentación formal del dataset que relacione labels con estructuras.

Se corrige explícitamente el label `209`, que aparece en SPIDER y debe incluirse en el grupo alto, no enviarse a fondo.


In [ ]:
# Mapping multiclase agrupado preliminar.
#
# Clase 0: fondo
# Clase 1: labels bajos 1..9
# Clase 2: label 100
# Clase 3: labels altos 201..209 y cualquier label observado >= 201

NUM_CLASSES = 4
LABEL_GROUP_MAPPING = {}

LABEL_GROUP_MAPPING[0] = 0

for label in range(1, 10):
    LABEL_GROUP_MAPPING[label] = 1

LABEL_GROUP_MAPPING[100] = 2

for label in range(201, 210):  # incluye 209
    LABEL_GROUP_MAPPING[label] = 3

# Si el análisis detecta otros labels altos, se agrupan automáticamente en clase 3.
for label in observed_original_labels:
    if int(label) >= 201:
        LABEL_GROUP_MAPPING[int(label)] = 3

LABEL_GROUP_MAPPING = {int(k): int(v) for k, v in LABEL_GROUP_MAPPING.items()}

covered_labels = set(LABEL_GROUP_MAPPING.keys())
unmapped_labels = sorted(set(observed_original_labels) - covered_labels)

if unmapped_labels:
    print("ADVERTENCIA: labels observados no mapeados. Se enviarán a fondo si aparecen:", unmapped_labels)
else:
    print("Todos los labels observados están cubiertos por el mapping.")

mapping_path = MULTICLASS_ROOT / "E5_multiclass_label_mapping.json"
with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(
        {str(k): int(v) for k, v in LABEL_GROUP_MAPPING.items()},
        f,
        indent=2,
        ensure_ascii=False,
    )

print("NUM_CLASSES:", NUM_CLASSES)
print("Mapping exportado en:", mapping_path)
print("Labels cubiertos:", sorted(LABEL_GROUP_MAPPING.keys()))
print("Clases destino:", sorted(set(LABEL_GROUP_MAPPING.values())))
print("Label 209 mapeado a:", LABEL_GROUP_MAPPING.get(209))


## 6. Modelo binario selector de slice

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleUNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)

        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)


binary_selector_model = None
slice_strategy_used = "central_slice_fallback"

if BINARY_MODEL_PATH.exists():
    try:
        binary_checkpoint = torch.load(BINARY_MODEL_PATH, map_location=DEVICE)
        binary_base_channels = int(binary_checkpoint.get("base_channels", 16))
        binary_selector_model = SimpleUNet2D(
            in_channels=1,
            out_channels=1,
            base_channels=binary_base_channels,
        ).to(DEVICE)

        binary_selector_model.load_state_dict(binary_checkpoint["model_state_dict"])
        binary_selector_model.eval()
        slice_strategy_used = "center_window_best_prediction"
        print("Modelo binario selector cargado:", BINARY_MODEL_PATH)
        print("base_channels binario:", binary_base_channels)
    except Exception as exc:
        binary_selector_model = None
        slice_strategy_used = "central_slice_fallback"
        print("No se pudo cargar modelo binario selector. Se usará central_slice_fallback.")
        print("Error:", repr(exc))
else:
    print("No existe modelo binario selector. Se usará central_slice_fallback.")

print("Estrategia de slice:", slice_strategy_used)


## 7. Funciones de preprocesamiento y dataset multiclase

In [ ]:
def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8):
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])
    clipped = np.clip(image_float, low, high)

    if np.isclose(high, low):
        normalized = np.zeros_like(clipped, dtype=np.float32)
    else:
        normalized = (clipped - low) / (high - low + eps)

    return normalized.astype(np.float32)


def take_slice(array, axis, index):
    return np.take(array, indices=int(index), axis=int(axis))


def resize_image_slice(image_slice, target_size=TARGET_SIZE):
    out = resize(
        image_slice,
        target_size,
        order=1,
        mode="constant",
        anti_aliasing=True,
        preserve_range=True,
    )
    return out.astype(np.float32)


def resize_mask_slice(mask_slice, target_size=TARGET_SIZE):
    out = resize(
        mask_slice,
        target_size,
        order=0,
        mode="constant",
        anti_aliasing=False,
        preserve_range=True,
    )
    return out.astype(np.int64)


def map_mask_to_grouped(mask):
    grouped = np.zeros_like(mask, dtype=np.int64)
    unique_labels = np.unique(mask)

    unmapped = []
    for label in unique_labels:
        label_int = int(label)

        if label_int in LABEL_GROUP_MAPPING:
            grouped[mask == label_int] = int(LABEL_GROUP_MAPPING[label_int])
        else:
            unmapped.append(label_int)
            grouped[mask == label_int] = 0

    if unmapped:
        print("ADVERTENCIA: labels no mapeados se enviaron a fondo:", sorted(set(unmapped)))

    return grouped.astype(np.int64)


def candidate_indices_for_center_window(n_slices, radius=CENTER_WINDOW_RADIUS):
    center = n_slices // 2
    start = max(0, center - radius)
    end = min(n_slices - 1, center + radius)
    return list(range(start, end + 1))


def select_slice_center_window_prediction(image_norm, axis=SAGITTAL_AXIS):
    n_slices = image_norm.shape[axis]

    if binary_selector_model is None:
        return n_slices // 2

    candidate_indices = candidate_indices_for_center_window(n_slices, radius=CENTER_WINDOW_RADIUS)

    tensors = []
    for idx in candidate_indices:
        image_slice = take_slice(image_norm, axis, idx)
        image_slice_resized = resize_image_slice(image_slice, TARGET_SIZE)
        tensors.append(torch.from_numpy(image_slice_resized).unsqueeze(0))

    batch = torch.stack(tensors, dim=0).to(DEVICE)

    with torch.no_grad():
        logits = binary_selector_model(batch)
        probs = torch.sigmoid(logits)
        foreground_scores = (probs >= 0.5).float().mean(dim=(1, 2, 3)).detach().cpu().numpy()

    best_local = int(np.argmax(foreground_scores))
    return int(candidate_indices[best_local])


def prepare_multiclass_sample(row):
    image_path, mask_path = get_case_paths(row)
    _, image = read_mha(image_path)
    _, mask = read_mha(mask_path)

    if tuple(image.shape) != tuple(mask.shape):
        raise ValueError(f"Shape incompatible: image={image.shape}, mask={mask.shape}")

    image_norm = robust_percentile_normalize(image)
    slice_index = select_slice_center_window_prediction(image_norm, axis=SAGITTAL_AXIS)

    image_slice = take_slice(image_norm, SAGITTAL_AXIS, slice_index)
    mask_slice_original = take_slice(mask, SAGITTAL_AXIS, slice_index)
    mask_slice_grouped = map_mask_to_grouped(mask_slice_original)

    image_resized = resize_image_slice(image_slice, TARGET_SIZE)
    mask_resized = resize_mask_slice(mask_slice_grouped, TARGET_SIZE)

    unique_values = np.unique(mask_resized)
    if unique_values.min() < 0 or unique_values.max() >= NUM_CLASSES:
        raise ValueError(f"Máscara agrupada fuera de rango: {unique_values}")

    return {
        "case_id": str(row["case_id"]),
        "image": image_resized.astype(np.float32),
        "mask": mask_resized.astype(np.int64),
        "slice_index": int(slice_index),
        "source_image_path": str(image_path),
        "source_mask_path": str(mask_path),
    }


samples = []

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Preparando muestras multiclase"):
    samples.append(prepare_multiclass_sample(row))

print("Muestras preparadas:", len(samples))
print("Ejemplo image:", samples[0]["image"].shape, samples[0]["image"].dtype)
print("Ejemplo mask:", samples[0]["mask"].shape, samples[0]["mask"].dtype)
print("Ejemplo mask unique:", np.unique(samples[0]["mask"]))


class MulticlassSliceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = torch.from_numpy(sample["image"]).unsqueeze(0).float()
        mask = torch.from_numpy(sample["mask"]).long()

        return {
            "image": image,
            "mask": mask,
            "case_id": sample["case_id"],
            "slice_index": sample["slice_index"],
            "source_image_path": sample["source_image_path"],
            "source_mask_path": sample["source_mask_path"],
        }


dataset = MulticlassSliceDataset(samples)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

split_rows = []
for split_name, subset in [("train", train_dataset), ("validation", val_dataset)]:
    for idx in subset.indices:
        sample = samples[idx]
        split_rows.append({
            "split": split_name,
            "case_id": sample["case_id"],
            "slice_index": sample["slice_index"],
            "source_image_path": sample["source_image_path"],
            "source_mask_path": sample["source_mask_path"],
        })

split_df = pd.DataFrame(split_rows)
split_csv_path = MULTICLASS_ROOT / "E5_multiclass_split.csv"
split_df.to_csv(split_csv_path, index=False)

batch = next(iter(train_loader))
print("Train cases:", len(train_dataset))
print("Validation cases:", len(val_dataset))
print("Batch image:", batch["image"].shape, batch["image"].dtype)
print("Batch mask:", batch["mask"].shape, batch["mask"].dtype)
print("Batch mask unique:", torch.unique(batch["mask"]))
print("CSV split:", split_csv_path)


## 8. Modelo, pesos de clase, pérdida y optimizador

In [ ]:
class SimpleUNet2DMulticlass(nn.Module):
    def __init__(self, in_channels=1, num_classes=4, base_channels=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        self.out_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)

        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)


def compute_class_weights_from_loader(loader, num_classes, clip_min=0.25, clip_max=5.0):
    pixel_counts = np.zeros(num_classes, dtype=np.float64)

    for batch in loader:
        masks_np = batch["mask"].detach().cpu().numpy()

        for cls in range(num_classes):
            pixel_counts[cls] += np.sum(masks_np == cls)

    total_pixels = np.sum(pixel_counts)

    if total_pixels == 0:
        weights = np.ones(num_classes, dtype=np.float32)
    else:
        frequencies = pixel_counts / total_pixels
        weights = 1.0 / np.maximum(frequencies, 1e-8)
        weights = weights / np.mean(weights)
        weights = np.clip(weights, clip_min, clip_max).astype(np.float32)

    return weights, pixel_counts


clipped_weights, class_pixel_counts = compute_class_weights_from_loader(
    train_loader,
    num_classes=NUM_CLASSES,
    clip_min=0.25,
    clip_max=5.0,
)

class_weights_path = MULTICLASS_ROOT / "E5_multiclass_class_weights.json"
with open(class_weights_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "class_pixel_counts": {str(i): int(v) for i, v in enumerate(class_pixel_counts.tolist())},
            "class_weights": {str(i): float(v) for i, v in enumerate(clipped_weights.tolist())},
            "note": "Pesos calculados desde train_loader y recortados para evitar valores extremos.",
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

weight_tensor = torch.tensor(clipped_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

multiclass_model = SimpleUNet2DMulticlass(
    in_channels=1,
    num_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
).to(DEVICE)

optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=LEARNING_RATE)

print("Modelo multiclase creado en:", DEVICE)
print("NUM_CLASSES:", NUM_CLASSES)
print("BASE_CHANNELS:", BASE_CHANNELS)
print("LEARNING_RATE:", LEARNING_RATE)
print("class_pixel_counts:", class_pixel_counts)
print("clipped_weights:", clipped_weights)
print("class weights exportados en:", class_weights_path)
print("criterion:", criterion)


## 9. Métricas multiclase y validación previa

In [ ]:
EPS = 1e-8

def multiclass_metrics_from_logits(logits, masks, num_classes=NUM_CLASSES):
    preds = torch.argmax(logits, dim=1)

    dice_by_class = {}
    iou_by_class = {}

    for cls in range(num_classes):
        pred_cls = preds == cls
        mask_cls = masks == cls

        intersection = torch.logical_and(pred_cls, mask_cls).sum().float()
        pred_sum = pred_cls.sum().float()
        mask_sum = mask_cls.sum().float()
        union = torch.logical_or(pred_cls, mask_cls).sum().float()

        dice = (2.0 * intersection + EPS) / (pred_sum + mask_sum + EPS)
        iou = (intersection + EPS) / (union + EPS)

        dice_by_class[cls] = float(dice.detach().cpu())
        iou_by_class[cls] = float(iou.detach().cpu())

    no_bg_classes = list(range(1, num_classes))
    dice_macro_no_bg = float(np.mean([dice_by_class[c] for c in no_bg_classes]))
    iou_macro_no_bg = float(np.mean([iou_by_class[c] for c in no_bg_classes]))

    pixel_accuracy = float((preds == masks).float().mean().detach().cpu())

    return {
        "dice_macro_no_bg": dice_macro_no_bg,
        "iou_macro_no_bg": iou_macro_no_bg,
        "pixel_accuracy": pixel_accuracy,
        "dice_by_class": dice_by_class,
        "iou_by_class": iou_by_class,
    }


batch = next(iter(train_loader))

images = batch["image"].to(DEVICE)
masks = batch["mask"].to(DEVICE)

print("images:", images.shape, images.dtype)
print("masks:", masks.shape, masks.dtype)
print("masks unique:", torch.unique(masks))

with torch.no_grad():
    logits = multiclass_model(images)

print("logits:", logits.shape, logits.dtype)
print("Esperado:", torch.Size([images.shape[0], NUM_CLASSES, 256, 256]))

assert images.shape[1] == 1, "La imagen debe tener un canal."
assert logits.shape[1] == NUM_CLASSES, "La salida del modelo no coincide con NUM_CLASSES."
assert masks.dtype == torch.long, "La máscara debe ser torch.long para CrossEntropyLoss."
assert int(torch.min(masks)) >= 0, "Hay labels negativos en la máscara."
assert int(torch.max(masks)) < NUM_CLASSES, "Hay labels fuera del rango 0..NUM_CLASSES-1."

metrics_preview = multiclass_metrics_from_logits(logits, masks)
print("Metrics preview:", metrics_preview)
print("Forward pass multiclase validado correctamente.")


## 10. Entrenamiento

In [ ]:
best_val_dice = -1.0
best_epoch = 0

best_model_path = MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_best.pt"
last_model_path = MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_last.pt"
history_csv_path = MULTICLASS_ROOT / "E5_multiclass_training_history.csv"

mapping_for_checkpoint = {
    str(int(k)): int(v)
    for k, v in LABEL_GROUP_MAPPING.items()
}

class_weights_for_checkpoint = (
    clipped_weights.tolist()
    if "clipped_weights" in globals()
    else [1.0 for _ in range(NUM_CLASSES)]
)


def run_epoch(loader, train=False):
    if train:
        multiclass_model.train()
    else:
        multiclass_model.eval()

    losses = []
    dice_values = []
    iou_values = []
    acc_values = []

    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)

            logits = multiclass_model(images)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            metrics = multiclass_metrics_from_logits(logits.detach(), masks)

            losses.append(float(loss.detach().cpu()))
            dice_values.append(float(metrics["dice_macro_no_bg"]))
            iou_values.append(float(metrics["iou_macro_no_bg"]))
            acc_values.append(float(metrics["pixel_accuracy"]))

    return {
        "loss": float(np.mean(losses)),
        "dice_macro_no_bg": float(np.mean(dice_values)),
        "iou_macro_no_bg": float(np.mean(iou_values)),
        "pixel_accuracy": float(np.mean(acc_values)),
    }


history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_dice_macro_no_bg": train_metrics["dice_macro_no_bg"],
        "train_iou_macro_no_bg": train_metrics["iou_macro_no_bg"],
        "train_pixel_accuracy": train_metrics["pixel_accuracy"],
        "val_loss": val_metrics["loss"],
        "val_dice_macro_no_bg": val_metrics["dice_macro_no_bg"],
        "val_iou_macro_no_bg": val_metrics["iou_macro_no_bg"],
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
    }

    history.append(row)
    print(row)

    if val_metrics["dice_macro_no_bg"] > best_val_dice:
        best_val_dice = val_metrics["dice_macro_no_bg"]
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": multiclass_model.state_dict(),
                "epoch": epoch,
                "val_dice_macro_no_bg": best_val_dice,
                "num_classes": NUM_CLASSES,
                "label_group_mapping": mapping_for_checkpoint,
                "class_weights": class_weights_for_checkpoint,
                "target_size": TARGET_SIZE,
                "sagittal_axis": SAGITTAL_AXIS,
                "base_channels": BASE_CHANNELS,
                "slice_strategy": slice_strategy_used,
            },
            best_model_path,
        )

torch.save(
    {
        "model_state_dict": multiclass_model.state_dict(),
        "history": history,
        "num_classes": NUM_CLASSES,
        "label_group_mapping": mapping_for_checkpoint,
        "class_weights": class_weights_for_checkpoint,
        "target_size": TARGET_SIZE,
        "sagittal_axis": SAGITTAL_AXIS,
        "base_channels": BASE_CHANNELS,
        "slice_strategy": slice_strategy_used,
        "best_epoch": best_epoch,
        "best_val_dice_macro_no_bg": best_val_dice,
    },
    last_model_path,
)

history_df = pd.DataFrame(history)
history_df.to_csv(history_csv_path, index=False)

print("Mejor epoch:", best_epoch)
print("Mejor val dice macro sin fondo:", best_val_dice)
print("Mejor modelo:", best_model_path)
print("Último modelo:", last_model_path)
print("History:", history_csv_path)

display(history_df)


## 11. Evaluación en validación

In [ ]:
# Cargar el mejor modelo antes de evaluar
checkpoint = torch.load(best_model_path, map_location=DEVICE)
multiclass_model.load_state_dict(checkpoint["model_state_dict"])
multiclass_model.eval()

case_metric_rows = []
class_metric_accumulator = {
    cls: {"dice": [], "iou": []}
    for cls in range(NUM_CLASSES)
}

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluando validación"):
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = multiclass_model(images)

        for i in range(images.shape[0]):
            case_id = batch["case_id"][i]
            slice_index = int(batch["slice_index"][i])

            single_logits = logits[i:i+1]
            single_mask = masks[i:i+1]
            metrics = multiclass_metrics_from_logits(single_logits, single_mask)

            row = {
                "case_id": case_id,
                "slice_index": slice_index,
                "dice_macro_no_bg": metrics["dice_macro_no_bg"],
                "iou_macro_no_bg": metrics["iou_macro_no_bg"],
                "pixel_accuracy": metrics["pixel_accuracy"],
                "source_image_path": batch["source_image_path"][i],
                "source_mask_path": batch["source_mask_path"][i],
            }

            for cls in range(NUM_CLASSES):
                row[f"dice_class_{cls}"] = metrics["dice_by_class"][cls]
                row[f"iou_class_{cls}"] = metrics["iou_by_class"][cls]
                class_metric_accumulator[cls]["dice"].append(metrics["dice_by_class"][cls])
                class_metric_accumulator[cls]["iou"].append(metrics["iou_by_class"][cls])

            case_metric_rows.append(row)

metrics_by_case_df = pd.DataFrame(case_metric_rows)
metrics_by_case_csv_path = MULTICLASS_ROOT / "E5_multiclass_metrics_by_case.csv"
metrics_by_case_df.to_csv(metrics_by_case_csv_path, index=False)

class_rows = []
for cls in range(NUM_CLASSES):
    class_rows.append({
        "class_id": cls,
        "dice_mean": float(np.mean(class_metric_accumulator[cls]["dice"])),
        "dice_std": float(np.std(class_metric_accumulator[cls]["dice"])),
        "iou_mean": float(np.mean(class_metric_accumulator[cls]["iou"])),
        "iou_std": float(np.std(class_metric_accumulator[cls]["iou"])),
        "n_cases": len(class_metric_accumulator[cls]["dice"]),
    })

metrics_by_class_df = pd.DataFrame(class_rows)
metrics_by_class_csv_path = MULTICLASS_ROOT / "E5_multiclass_metrics_by_class.csv"
metrics_by_class_df.to_csv(metrics_by_class_csv_path, index=False)

summary_df = pd.DataFrame([{
    "best_epoch": int(best_epoch),
    "val_dice_macro_no_bg": float(metrics_by_case_df["dice_macro_no_bg"].mean()),
    "val_iou_macro_no_bg": float(metrics_by_case_df["iou_macro_no_bg"].mean()),
    "val_pixel_accuracy": float(metrics_by_case_df["pixel_accuracy"].mean()),
    "n_validation_cases": len(metrics_by_case_df),
}])
summary_csv_path = MULTICLASS_ROOT / "E5_multiclass_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Métricas por caso:", metrics_by_case_csv_path)
print("Métricas por clase:", metrics_by_class_csv_path)
print("Summary:", summary_csv_path)
display(summary_df)
display(metrics_by_class_df)
display(metrics_by_case_df.head())


## 12. Visualizaciones

In [ ]:
def plot_multiclass_example(sample_item, output_path):
    image = sample_item["image"]
    gt = sample_item["mask"]

    x = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        logits = multiclass_model(x)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.int64)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Imagen")

    axes[1].imshow(gt, vmin=0, vmax=NUM_CLASSES-1)
    axes[1].set_title("GT agrupada")

    axes[2].imshow(pred, vmin=0, vmax=NUM_CLASSES-1)
    axes[2].set_title("Predicción")

    axes[3].imshow(image, cmap="gray")
    axes[3].imshow(np.ma.masked_where(gt == 0, gt), alpha=0.45, vmin=0, vmax=NUM_CLASSES-1)
    axes[3].set_title("Overlay GT")

    axes[4].imshow(image, cmap="gray")
    axes[4].imshow(np.ma.masked_where(pred == 0, pred), alpha=0.45, vmin=0, vmax=NUM_CLASSES-1)
    axes[4].set_title("Overlay pred")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(f"Multiclase agrupado - {sample_item['case_id']} - slice {sample_item['slice_index']}")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    return output_path


# Usar hasta 3 muestras de validación para evidencia visual
validation_indices = list(val_dataset.indices)[:3]
figure_paths = []

for idx, sample_idx in enumerate(validation_indices, start=1):
    sample_item = samples[sample_idx]
    output_path = FIGURES_ROOT / f"E5_multiclass_prediction_example_{idx:02d}.png"
    plot_multiclass_example(sample_item, output_path)
    figure_paths.append(str(output_path))

# Curvas de entrenamiento
loss_curve_path = FIGURES_ROOT / "E5_multiclass_loss_curve.png"
dice_curve_path = FIGURES_ROOT / "E5_multiclass_dice_curve.png"

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
plt.title("E5 multiclase - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_dice_macro_no_bg"], marker="o", label="train_dice_macro_no_bg")
plt.plot(history_df["epoch"], history_df["val_dice_macro_no_bg"], marker="o", label="val_dice_macro_no_bg")
plt.title("E5 multiclase - Dice macro sin fondo")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(dice_curve_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figuras predicción:", figure_paths)
print("Loss curve:", loss_curve_path)
print("Dice curve:", dice_curve_path)


## 13. Comparación contextual contra binario

In [ ]:
binary_context_df = pd.DataFrame([{
    "binary_holdout_dice": 0.8816,
    "binary_holdout_iou": 0.7904,
    "multiclass_val_dice_macro_no_bg": float(summary_df["val_dice_macro_no_bg"].iloc[0]),
    "multiclass_val_iou_macro_no_bg": float(summary_df["val_iou_macro_no_bg"].iloc[0]),
    "note": "No son métricas directamente equivalentes: binario mide foreground/background; multiclase mide separación entre grupos técnicos.",
}])

binary_context_csv_path = MULTICLASS_ROOT / "E5_multiclass_vs_binary_context.csv"
binary_context_df.to_csv(binary_context_csv_path, index=False)

print("Comparación contextual:", binary_context_csv_path)
display(binary_context_df)


## 14. Reporte JSON y conclusión técnica

In [ ]:
report_path = MULTICLASS_ROOT / "E5_multiclass_validation_report.json"
conclusion_path = DOCS_ROOT / "E5_multiclase_agrupado_conclusion.md"

dice_by_class = {
    str(int(row["class_id"])): float(row["dice_mean"])
    for _, row in metrics_by_class_df.iterrows()
}
iou_by_class = {
    str(int(row["class_id"])): float(row["iou_mean"])
    for _, row in metrics_by_class_df.iterrows()
}

report = {
    "n_cases": int(len(selected_df)),
    "train_cases": int(len(train_dataset)),
    "validation_cases": int(len(val_dataset)),
    "modality": MODALITY_FILTER,
    "num_classes": int(NUM_CLASSES),
    "label_group_mapping": {str(int(k)): int(v) for k, v in LABEL_GROUP_MAPPING.items()},
    "epochs": int(EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "device": str(DEVICE),
    "class_weights": {str(i): float(v) for i, v in enumerate(clipped_weights.tolist())},
    "best_epoch": int(best_epoch),
    "val_dice_macro_no_bg": float(summary_df["val_dice_macro_no_bg"].iloc[0]),
    "val_iou_macro_no_bg": float(summary_df["val_iou_macro_no_bg"].iloc[0]),
    "val_pixel_accuracy": float(summary_df["val_pixel_accuracy"].iloc[0]),
    "dice_by_class": dice_by_class,
    "iou_by_class": iou_by_class,
    "slice_strategy": slice_strategy_used,
    "exports": {
        "selected_cases": str(selected_cases_csv_path),
        "original_label_distribution": str(original_label_distribution_csv_path),
        "label_mapping": str(mapping_path),
        "class_weights": str(class_weights_path),
        "split": str(split_csv_path),
        "history": str(history_csv_path),
        "metrics_by_case": str(metrics_by_case_csv_path),
        "metrics_by_class": str(metrics_by_class_csv_path),
        "summary": str(summary_csv_path),
        "model_best": str(best_model_path),
        "model_last": str(last_model_path),
        "figures": figure_paths,
        "loss_curve": str(loss_curve_path),
        "dice_curve": str(dice_curve_path),
        "binary_context": str(binary_context_csv_path),
    },
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

class_table_md = metrics_by_class_df.to_markdown(index=False)
summary_table_md = summary_df.to_markdown(index=False)

conclusion_text = f"""# Conclusión técnica - E5 Baseline multiclase agrupado sagital

## Objetivo

Se construyó un primer baseline multiclase 2D sagital sobre SPIDER, partiendo del pipeline binario validado. El objetivo fue evaluar si una U-Net 2D simple puede diferenciar grupos técnicos de etiquetas, en lugar de segmentar únicamente foreground/background.

## Configuración

- Dataset: SPIDER.
- Modalidad: `{MODALITY_FILTER}`.
- Casos seleccionados: {len(selected_df)}.
- Train: {len(train_dataset)}.
- Validación: {len(val_dataset)}.
- Eje sagital: {SAGITTAL_AXIS}.
- Target size: {TARGET_SIZE}.
- Número de clases: {NUM_CLASSES}.
- Estrategia de slice: `{slice_strategy_used}`.
- Modelo: U-Net 2D simple multiclase.
- Pérdida: CrossEntropyLoss con pesos de clase calculados desde el conjunto de entrenamiento.
- Device: `{DEVICE}`.

## Mapping agrupado utilizado

El mapping es técnico y preliminar. No asigna nombres anatómicos definitivos sin documentación formal del dataset.

- Clase 0: fondo.
- Clase 1: labels bajos 1..9.
- Clase 2: label 100.
- Clase 3: labels altos 201..209 y otros labels observados >= 201.

## Resultados principales

{summary_table_md}

## Resultados por clase

{class_table_md}

## Comparación conceptual contra binario

El pipeline binario obtuvo previamente un Dice holdout aproximado de 0.8816 e IoU de 0.7904. Estos valores no son directamente equivalentes al multiclase, porque el binario evalúa foreground/background, mientras que el multiclase evalúa separación entre grupos técnicos.

## Interpretación

El baseline multiclase agrupado permite evaluar la viabilidad de pasar de una segmentación binaria a una segmentación por grupos de etiquetas. Si alguna clase presenta Dice cercano a cero, se debe considerar ajuste de mapping, balanceo de clases, sampling por clase o mayor cantidad de datos.

## Limitaciones

- Mapping agrupado preliminar.
- No se asignan nombres anatómicos definitivos.
- Entrenamiento 2D sobre un slice sagital.
- No inferencia 3D.
- No axial.
- No nnU-Net.
- Posible desbalance entre clases.
- Validación interna, no clínica.

## Recomendación próxima

Revisar métricas por clase y visualizaciones. Si una clase queda con bajo desempeño, conviene diagnosticar distribución/sampling antes de avanzar a multiclase completa, axial o nnU-Net.
"""

with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Reporte JSON:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Conclusión Markdown:", conclusion_path)
print(conclusion_text)
